# 6. Análisis Específico: La Crisis de las "Muertes por Desesperación"

Como se discutió en el marco teórico, el aumento de suicidios y accidentes (especialmente sobredosis) ha sido uno de los fenómenos más preocupantes en la mortalidad de EE.UU. durante el período de estudio.

## 6.1. Evolución conjunta de suicidio y accidentes

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("NCHS_Leading_Causes.csv", dtype=str)
df.columns = ['year','cause_113','cause_name','state','deaths','age_adjusted_death_rate']
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['deaths'] = pd.to_numeric(df['deaths'].str.replace('.','',regex=False).str.replace(',','',regex=False), errors='coerce').fillna(0).astype(int)
df['age_adjusted_death_rate'] = pd.to_numeric(df['age_adjusted_death_rate'].str.replace(',','.'), errors='coerce')
df['state'] = df['state'].str.strip()
df['cause_name'] = df['cause_name'].str.strip()
df = df[(df['year']>=1999)&(df['year']<=2017)].dropna(subset=['year','age_adjusted_death_rate'])

estados = df[df['state']!='United States']
us = df[df['state']=='United States']
estados_2017 = estados[estados['year']==2017]
sin_all = estados[estados['cause_name']!='All causes']
print(f"Datos cargados: {len(df):,} filas | {df['state'].nunique()} entidades | {df['year'].min():.0f}–{df['year'].max():.0f}")


Datos cargados: 10,840 filas | 52 entidades | 1999–2017


In [2]:
causas = ['Suicide','Unintentional injuries']
d = nac = estados[estados['cause_name'].isin(causas)].groupby(['year','cause_name'])['deaths'].sum().reset_index()

colors_map = {'Suicide':'#2F4F4F','Unintentional injuries':'#528B8B'}

fig = go.Figure()
for causa in causas:
    sub = d[d['cause_name']==causa].sort_values('year')
    fig.add_trace(go.Scatter(
        x=sub['year'], y=sub['deaths'], mode='lines+markers',
        name=causa, line=dict(color=colors_map[causa], width=2),
        marker=dict(size=6),
        hovertemplate=f'<b>{causa}</b><br>Año: %{{x}}<br>Muertes: %{{y:,}}<extra></extra>'
    ))

fig.add_vline(x=2014, line_dash='dot', line_color='gray',
              annotation_text='2014: aceleración', annotation_position='top left')
fig.update_layout(
    title="Evolución de las 'Muertes por Desesperación' (1999–2017)",
    xaxis_title='Año', yaxis_title='Número de muertes',
    height=470, template='plotly_white', hovermode='x unified',
    xaxis=dict(tickmode='linear', tick0=1999, dtick=1),
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

**Interpretación:** Tanto el suicidio como las lesiones no intencionales presentan una tendencia ascendente, con un incremento más marcado a partir de 2014. Las muertes por lesiones no intencionales muestran una aceleración pronunciada vinculada a la epidemia de opioides. Este comportamiento contrasta con la reducción observada en otras causas de mortalidad y evidencia un cambio en los determinantes sociales y de salud.